# Laboratório 8: Segmentação Morfológica Watershed

**Equipe:** 
* Eduardo de Souza Carrilho - RA: 11201812084  
* Gabriel Figueiredo de Souza - RA: 11202230332  

**Data do Experimento:** 14/04/2026  
**Data de Entrega:** 28/04/2026

---

## 1. Introdução

A segmentação é uma das etapas mais críticas no processamento digital de imagens, sendo responsável por particionar uma imagem em suas partes constituintes ou objetos de interesse. Neste contexto, o algoritmo *Watershed* (divisor de águas) é uma ferramenta clássica e poderosa da morfologia matemática computacional, especialmente útil na separação de objetos que se tocam ou estão sobrepostos. 

Neste laboratório, o foco foi a exploração teórica e prática da segmentação por Watershed aliada a processos de limiarização, rotulação (*labeling*), cálculos de distâncias (*Distance Transform*) e filtragens espaciais para suavização. O objetivo final culminou na construção de uma rotina completa capaz de isolar, identificar e até extrair objetos específicos de uma cena, com aplicações direcionadas à segmentação dos integrantes do grupo e seus respectivos avatares.

---

## 2. Procedimentos Experimentais

A condução do laboratório foi dívida entre a observação passo a passo do algoritmo clássico de segmentação de moedas do OpenCV, e a posterior replicação desses conceitos nas imagens da equipe. As etapas aplicadas abrangeram:

* **Filtragem Espacial e Remoção de Ruído:** Foram testadas diferentes máscaras de suavização — Filtros lineares (blur, Gaussian Blur, box) e não-lineares (Filtro Mediano e Filtro Bilateral). Esses procedimentos buscaram amenizar o ruído da imagem original preservando ao máximo as bordas lógicas das superfícies.
* **Binarização com Thresholding de Otsu:** Para preparar a imagem para análise geométrica, foi aplicado o processo de Otsu associado ao limiar invertido, separando os pixels de fundo (intensidade baixíssima) dos possíveis objetos.
* **Componentes Básicos da Segmentação (Sure BG e Sure FG):**
  * **Sure Background (Fundo Seguro):** Utilizou-se operação de *Abertura* seguida por forte *Dilatação* para garantir que uma região pertencia inteiramente ao fundo.
  * **Sure Foreground (Primeiro Plano Seguro):** Extraído a partir da **Transformada de Distância**, que quantifica a distância geométrica das bordas de um objeto binarizado. Esse passo foi limiarizado para encontrar os picos/centros isolados de cada objeto, que certamente compõem seu núcleo.
  * **Unknown Region (Região Desconhecida):** Subtração entre Sure BG e Sure FG, formando o limítrofe onde as bordas exatas deveriam ser buscadas pelo Watershed.
* **Rotulação e Watershed:** Uso intenso dos métodos *Connected Components* (e o implementado `mm.label()`) para rotular os centros das regiões. Estes marcadores alimentaram o `cv2.watershed()`, guiando os pontos de "enchente" topográfica até que as bacias se encontrassem e delimitassem as fronteiras ótimas (-1) entre os fragmentos em análise.
* **Aplicações (3.a, 3.b e 3.c):** Implementação de programas que consolidam estes passos em: imagens dos alunos, dos avatares, superposição de boundings ou coloração dos contornos, e elaboração de um pipe contínuo com WebCam em tempo real.

---

## 3. Análise e Discussão dos Estudos Realizados

Durante a execução da primeira etapa didática — a extração de bordas na imagem teste (moedas ou grãos) —, as principais compreensões técnicas adquiridas foram:

**a) Comparativo de Suavização (Filtros):** A passagem por `filter2D` e `blur` padrão produziram um desfoque homogêneo que suja as bordas. Já o filtro **Bilateral**, em contraposição aos Gausianos convencionais, teve notável desempenho porque considerou além da dimensão espacial dimensional a diferença fotométrica dos vizinhos; desta forma ele lavou/limpou imperfeições do preenchimento da superfície da moeda sem borrar severamente as suas transições estruturais. 

**b) Thresholding + Operação Morfológica (Opening):** O limiar de Otsu foi excelente na separação macro, porém o limiar em si resulta em muitos minúsculos pólos ruídosos que restaram de irregularidades de sombra e detritos. A **Abertura Morfológica** (`cv_MORPH_OPEN`) atendeu magistralmente no enxugamento desses polos porque desgastou esse ruído solto (completando fundos escuros de forma limpa) sem corroer a área total macro das superfícies das moedas.

**c) Transformada de Distância e "Sure FG":** Esta é a essência para desatar objetos fortemente sobrepostos (quando o blob do Otsu se grudou). Pela *Distance Transform*, os contornos viraram declives rasos de valores, mas os epicentros de geometria formaram cumes altos. O thresholding deste cume criou marcadores separados unicamente de cada centro de massa. O que unificava duas moedas coladas passou a ser visualizado apenas em seus distantes núcleos. Isso foi a mágica central da rotina que permitiu encontrar que efetivamente eram 'n' moedas invés de uma massaroca genérica.

**d) Watershed:** É o algoritmo responsável por transformar a marcação "Sure FG / BG", subindos os níveis a partir dos marcadores (bacias/vales), até criar os divisores de águais exatos localizados na *Unknown Region*. Sua topografia preencheu o que os limitadores não englobavam antes.

**e) Análise sobre as Questões Laboratoriais (3.a, 3.b e 3.c):**   
Na tentativa de fragmentar objetos complexos como pessoas em um avatar e contorno real, foi percebido que texturas complexas formam "falsas" bacias se o limiar não estiver hiper-suavizado. 
* Em imagens complexas, notou-se um "over-segmentation". O Watershed falha brutalmente se o *Sure FG* tiver múltiplos pontos fortes (devido aos vincos das roupas ou cores fortes dos avatares), porcionando o corpo inteiro. Deste modo, o Filtro Gaussiano de núcleo grande ou MedianBlur agressivo antes da limiarização demonstrou ser prioritário nesses perfis.
* O processo rodado na Webcam adiciona contornos temporais instáveis; o distúrbio natural das frestas de luz na cena fez as fronteiras "sambarem". As bordas morfológicas aplicadas intermitentemente com identificadores evidenciaram as limitações mecânicas da técnica versus técnicas robustas em Deep Learning modernas. A constante variação de luminosidade da webcam provocou ruídos mutáveis.

---

## 4. Conclusões

O Laboratório 8 forneceu sólidos alicerces a respeito das interdependências das etapas num *pipeline* canônico de Processamento Digital de Imagens clássico. Desvendamos na prática como falhas minúsculas numa etapa primária — de um simples filtro não utilizado para limpeza — tem um efeito dominó de corromper o processo rotular secundário, por criar excesso de sementes marcadoras (causando excesso severo de particionamento e sobre-segmentação pelo método *watershed*).

Para as premissas deste projeto final, a assimilação de operações morfológicas em tempo real na webcam revelou as maravilhas da performance morfológica frente as severas instabilidades de luminosidade das capturas diárias; provando que métodos baseados em Threshold base não-supervisionados impõem duras restrições técnicas no monitoramento de detecções biológicas se não adequadamente adaptadas e balanceadas ao limite visual local.

---

## 5. Referências Consultadas e Indicadas

* **Gonzalez, R. C., & Woods, R. E.** (2018). *Digital Image Processing* (4.ª ed.). Pearson. 
* **Tutorial OpenCV Python:** *Image Segmentation with Watershed Algorithm*. Documentação Oficial. Disponível em: https://docs.opencv.org/4.x/d3/db4/tutorial_py_watershed.html
* Material Didático e Exercícios: Notebook `pdi26_lab8_segmentacao_.ipynb` da UFABC.